# Enrichment Analysis

This notebook demonstrates three complementary enrichment analyses on the toy `protocol_example` dataset, each a self-contained part with its own inputs and outputs: **EOO** (Enrichment Of Overlap via block jackknife), **Pathway** enrichment (KEGG / clusterProfiler), and **sLDSC** (stratified LD-score regression with PolyFun).


## 1. EOO — Enrichment Of Overlap (block jackknife)

EOO evaluates the enrichment of significant variants within genomic annotations. Using a leave-one-chromosome-out (LOCO) block-jackknife approach it estimates the **Odds Ratio (OR)** and **Enrichment** for each annotation column, with a mean statistic and standard error aggregated across chromosomes. The OR is the odds of a variant being significant given an annotation is present versus absent; Enrichment is the proportion of significant variants within an annotation relative to its background proportion.


### Input Files

| File | Used as |
| --- | --- |
| significant variants file (`--significant_variants_path`) | RDS/TSV/TXT with `chr`, `pos` of significant variants |
| baseline annotation file (`--baseline_anno_path`) | tabular file with `CHR`, `BP`, and binary annotation columns (0/1) from the 7th column onward |


### **Step 1.** Run the EOO enrichment pipeline


In [ ]:
sos run pipeline/eoo_enrichment.ipynb enrichment \
    --significant_variants_path data/eoo_enrichment/colocboost_binary_vcp0.1_hg38_annotation.tsv.gz \
    --baseline_anno_path data/eoo_enrichment/colocboost_binary_vcp0.1_hg38_annotation_data.tsv \
    --name enrichment_results \
    --cwd output/eoo_enrichment

### Inspect the EOO results


In [ ]:
# R CODE
setwd('/home/ubuntu/xqtl_protocol_exercise')
library(data.table)
eoo_results = fread('output/eoo_enrichment/enrichment/enrichment_results.enrichment_results_summary.tsv.gz')
head(eoo_results)

### Output Files

An `.RDS` file (and a `enrichment_results_summary.tsv.gz` summary) containing a `summary` data.frame with per-annotation Odds Ratio, Enrichment, mean statistic and standard error across chromosomes.


## 2. Pathway enrichment (KEGG / clusterProfiler)

Pathway enrichment identifies biological pathways that are statistically over-represented in a gene set, helping infer biological functions, disease relevance, or regulatory mechanisms. The analysis uses the **clusterProfiler** R package with the KEGG database: genes are subset by group, a KEGG hypergeometric test is run per group, and pathway-level statistics are computed (raw and FDR-adjusted p-values, fold enrichment, RichFactor, z-score). Key parameters: `organism = "hsa"` (human) and `pvalue_cutoff = 1` (significance threshold for filtering).


### Input Files

| File | Used as |
| --- | --- |
| `data/pathway/test_pathway_genes_input.tsv` | gene list with group labels (columns `group`, `gene_id`) |


### **Step 1.** Run the pathway (GSEA/KEGG) pipeline


In [ ]:
sos run pipeline/gsea.ipynb pathway_analysis \
    --genes_file data/pathway/test_pathway_genes_input.tsv \
    --cwd output/pathway_results \
    --name test_genes

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/gsea.ipynb pathway_analysis \
    --genes_file data/pathway/test_pathway_genes_input.tsv \
    --cwd output/pathway_results --name test_genes

### Inspect the pathway results


In [ ]:
# r code
pathway_result = readRDS('output/pathway_results/pathway_analysis/test_genes.pathway_results.rds')
head(pathway_result)

### Output Files

A table with one row per enriched pathway per gene group. Columns: `ID` (KEGG pathway id), `Description`, `GeneRatio`, `BgRatio`, `RichFactor`, `FoldEnrichment`, `zScore`, `pvalue`, `p.adjust` (FDR), `Count`, and `group`.


## 3. sLDSC enrichment (PolyFun)

Stratified LD-score regression partitions trait heritability across functional annotations using PolyFun. The workflow has three stages: (1) build annotation files and compute LD-scores for the provided SNP list; (2) run LDSC regression to estimate per-annotation heritability for each GWAS sumstat; (3) summarise results per trait and run a meta-analysis across GWAS groups. The meta-analysis reports `tau` (the annotation's independent contribution to heritability, or fold-enrichment), its standard deviation `SD`, and the meta-analysed p-value `P`.


### Input Files

| File | Used as |
| --- | --- |
| `data/polyfun/input/colocboost_test_annotation_path.txt` | annotation file for the SNP list |
| `data/polyfun/input/reference_annotation0.txt` | reference annotation |
| `data/polyfun/input/genome_reference_bfile.txt` | genome reference bfile list |
| `data/polyfun/example_data` | LD reference / sumstats / weights example data |
| `data/polyfun/input/sumstats_test_all.txt` | list of GWAS sumstats (all traits) |
| `data/github/polyfun` | PolyFun toolkit (`--polyfun_path`) |


### **Step 1.** Make annotation files and calculate LD-scores


In [ ]:
sos run pipeline/sldsc_enrichment.ipynb make_annotation_files_ldscore \
    --annotation_file  data/polyfun/input/colocboost_test_annotation_path.txt \
    --reference_anno_file data/polyfun/input/reference_annotation0.txt \
    --genome_ref_file data/polyfun/input/genome_reference_bfile.txt \
    --annotation_name test_colocboost \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --cwd output/polyfun/ -j 22
 

In [ ]:
# step 1, make annotation of provided snp list, and calculate ldscore
 sos run pipeline/sldsc_enrichment.ipynb make_annotation_files_ldscore \
    --annotation_file  data/polyfun/input/colocboost_test_annotation_path.txt \
    --reference_anno_file data/polyfun/input/reference_annotation0.txt \
    --genome_ref_file data/polyfun/input/genome_reference_bfile.txt \
    --annotation_name test_colocboost \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --cwd output/polyfun/ -j 22
 

### **Step 2.** LDSC regression: calculate heritability


In [ ]:
sos run pipeline/sldsc_enrichment.ipynb get_heritability \
    --maf_cutoff 0 \
    --target_anno_dirs output/polyfun/test_colocboost_single_1 \
    --sumstat_dir data/polyfun/example_data \
    --baseline_ld_dir data/polyfun/example_data \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --weights_dir data/polyfun/example_data \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --annotation_name test_colocboost \
    --cwd output/polyfun/test_colocboost/sumstats/ \
    --all_traits_file data/polyfun/input/sumstats_test_all.txt \
    -s build -j 2


In [ ]:
# step 2 ldsc regression: calculate heritability 
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/sldsc_enrichment.ipynb get_heritability \
    --target_anno_dir output/polyfun/test_colocboost \
    --sumstat_dir data/polyfun/example_data \
    --baseline_ld_dir data/polyfun/example_data \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --weights_dir data/polyfun/example_data \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --annotation_name test_colocboost \
    --cwd output/polyfun/test_colocboost/sumstats/ \
    --all_traits_file data/polyfun/input/sumstats_test_all.txt \
    -s build -j 2

### **Step 3.** Summarise per trait and meta-analyse across GWAS groups


In [ ]:
sos run pipeline/sldsc_enrichment.ipynb postprocess \
    --traits_file data/polyfun/input/sumstats_test_all.txt \
    --heritability_cwd output/polyfun/test_colocboost/sumstats \
    --target_categories ANNOT_0 \
    --target_categories_label test_colocboost_annotation \
    --target_anno_dir output/polyfun/test_colocboost_single_1 \
    --annotation_name test_colocboost \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --maf_cutoff 0 \
    --cwd output/polyfun/test_colocboost/postprocess -j 4

In [ ]:
# step 3: summarize results for each gwas trait, and implement meta analysis across diff gwas group
sos run pipeline/sldsc_enrichment.ipynb processed_stats \
    --target_anno_dir output/polyfun/test_colocboost \
    --sumstat_dir data/polyfun/example_data \
    --baseline_ld_dir data/polyfun/example_data \
    --python_exec python \
    --polyfun_path data/github/polyfun \
    --weights_dir data/polyfun/example_data \
    --plink_name reference. \
    --baseline_name annotations. \
    --weight_name weights. \
    --annotation_name test_colocboost \
    --cwd output/polyfun/test_colocboost/sumstats/ \
    --trait_group_paths "data/polyfun/input/sumstats_test_all.txt data/polyfun/input/sumstats_test_category1.txt" \
    --trait_group_names "All category1" \
    --all_traits_file data/polyfun/input/sumstats_test_all.txt \
    -s force   


### Inspect per-GWAS sLDSC results


In [ ]:
# output
setwd('/home/ubuntu/xqtl_protocol_exercise')
sldsc_each_gwas_results = readRDS('output/polyfun/test_colocboost/sumstats/test_colocboost.single_tau.initial_processed_stats.rds')
str(sldsc_each_gwas_results)

### Inspect the meta-analysis results


In [ ]:
# meta analysis results across traits group output
setwd('/home/ubuntu/xqtl_protocol_exercise')
sldsc_meta_results = readRDS('output/polyfun/test_colocboost/sumstats/processed_stats_2/single_tau.test_colocboost.meta_processed_stats.rds')
sldsc_meta_results

### Output Files

Per-GWAS results: `output/polyfun/.../single_tau.initial_processed_stats.rds`. Meta-analysis: `output/polyfun/.../single_tau.<name>.meta_processed_stats.rds`, with columns `tau`/`Mean`, `SD`, and `P` per annotation.
